# CS 3892 / 5892 — Session 20 · Lean 4 II: transition systems in Lean

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-11-05-lean-4-transition-systems.ipynb)

**Thursday, November 5, 2026.** A transition system in Lean is a structure with an `init` and a
`step`; `Reachable` is an inductive predicate, so a fact about every reachable state is proved by
**induction on how the state was reached**. That induction is why an **inductive invariant** holds.
Then session 9's add-2 machine, where `x ≠ 5` is **true but not inductive**, and the **stronger**
invariant that proves it; the running counter with a power switch, proved for **every cap `N`**; and
how to **build** a reachable state, to show the bound is tight.

The definitions (`TS`, `Reachable`, `InductiveInvariant`, `Invariant`, `inductive_invariant_holds`)
are the ones HW4's `Defs.lean` gives you, with the same names. Every Lean snippet on the slides is a
region of a file in `lean/S20/`, and every output and goal the slides show is asserted by a script
in `python/`. Lean v4.33.0, core library only. Colab has no Lean editor, so the scripts print the
goals; to see them as you type, open `lean/` in VS Code with the Lean 4 extension.

## Setup

Run these two once. The second installs Lean: give it a minute or two.

In [ ]:
# --- Setup: find the repo (clone on Colab), define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-11-05-lean-4-transition-systems"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

PYD = ROOT / "sessions" / SESSION / "python"
LD  = ROOT / "sessions" / SESSION / "lean"
FIG = ROOT / "sessions" / SESSION / "figures"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- Lean v4.33.0, installed by HW4's checker: elan (the Lean version manager, pinned by
# SHA-256) into ~/.elan, then Lean through it -- about 575 MB, a minute or two the first time,
# and nothing to install after that. A Lean you already have is used as it is.
out = run(PYD / "lean_cli.py")
assert "version 4.33.0" in out

## 1. Reachable, and why an inductive invariant holds

`Reachable` has two constructors: an initial state, or one step from a reachable state. So
`inductive_invariant_holds` is an induction with two cases, and the script prints Lean's goal in
each.

In [ ]:
out = run(PYD / "01_reachable_and_induction.py")
assert "an inductive invariant holds in every reachable state" in out

## 2. The counter with a power switch: every step, every cap `N`

The state is a structure (`on`, `x`); a step is an inductive relation with one constructor per
transition. `cases` on the step gives one goal per transition. The proof never names a value of
`N`, so it covers every cap at once: what nuXmv, one cap at a time, cannot do.

In [ ]:
display(SVG(filename=str(FIG / "switch.svg")))
out = run(PYD / "02_switch_every_step.py")
assert "for every cap N" in out

## 3. True, and not inductive: session 9's add-2, in Lean

Start at 0, add 2 forever. `x ≠ 5` holds in every reachable state, but 3 satisfies it and steps
to 5. In Lean, "not inductive" is a theorem.

In [ ]:
display(SVG(filename=str(FIG / "add2.svg")))
out = run(PYD / "03_true_but_not_inductive.py")
assert "3 is never reached" in out

## 4. Strengthening: prove "x is even", then derive `x ≠ 5`

The stronger invariant admits fewer states, and that is what makes it inductive. HW4 Part 3 asks
for the same move on a different system.

In [ ]:
out = run(PYD / "04_strengthen.py")
assert "P holds in every reachable state" in out

## 5. Building reachable states: the bound is tight

`reach_k` builds `⟨true, k⟩` for every `k ≤ N`, one `Reachable.step` at a time, by induction on
`k`. `bound_tight` uses it as the witness of an `∃`.

In [ ]:
out = run(PYD / "05_build_reachable_states.py")
assert "it is tight" in out

## 6. The whole project builds, on the standard axioms

`lake build`, as VS Code runs it, then `#print axioms` for every theorem: no `sorry`, and nothing
classical.

In [ ]:
show(LD / "S20.lean")
out = run(PYD / "06_whole_library.py")
assert "Build completed successfully" in out and "Quot.sound" in out

## Try it yourself

Open the folder icon (left) → `cs3892-examples/sessions/cs3892-2026-11-05-lean-4-transition-systems/lean/S20/`,
double-click a file, edit, save (Ctrl+S), and re-run its cell. A runtime is deleted when it
disconnects: download anything you want to keep.

- In `Add2.lean`, change the step to `y = x + 3`. Which theorem breaks? Is `x ≠ 5` still true?
  Find a stronger invariant that proves whatever is true now.
- In `Switch.lean`, add a fifth transition, `reset : CtrStep N ⟨true, x⟩ ⟨true, 0⟩`. Which proof
  needs a new case, and what is its goal?
- In `Switch.lean`, try to prove `x ≤ N - 1` instead of `x ≤ N`. Where does it fail, and which
  theorem tells you it must?